In [1]:
import numpy as np 

In [ ]:
class Decision_Tree:

    def __init__(self,max_depth,min_samples_split):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.tree = None

    def entropy(self,y):
        _, counts = np.unique(y, return_counts=True)
        p = counts / len(y)
        return -np.sum(p * np.log2(p))

    def gini(self,y):

        _,counts = np.unique(y,return_counts = True)
        p = counts/len(y)
        p2 = np.sum(p**2)
        return 1 - p2 

    def weighted_impurity(self,y_left,y_right):

        n = len(y_left) + len(y_right)

        if n == 0:
            return 0

        left_weight = len(y_left)/n
        right_weight = len(y_right)/n
        return left_weight * self.gini(y_left) + right_weight * self.gini(y_right)

    def find_best_split(self,X,y):

        total_loss = float('inf')
        best_feature = None
        best_threshold = None

        parent_impurity = self.gini(y)

        for col in range(X.shape[1]):
            val= np.unique(X[:,col])

            if len(val)<2:
                continue 

            thresholds = (val[:-1] + val[1:]) / 2

            for threshold in thresholds:
                left_mask = X[:,col]<=threshold
                right_mask = ~left_mask

                y_left = y[left_mask]
                y_right = y[right_mask]

                loss = self.weighted_impurity(y_left,y_right)


                if loss < total_loss:
                    total_loss = loss
                    best_feature = col 
                    best_threshold = threshold

        if best_feature is None:
            return None,None,0
        best_gain  = parent_impurity - total_loss

        return best_feature,best_threshold,best_gain

    def build_tree(self,X,y,depth=0):

        classes, counts = np.unique(y, return_counts=True)
        prediction = classes[np.argmax(counts)]


        if len(classes) == 1:
            return {
                "leaf":True,
                "prediction":prediction
            }

        if self.max_depth is not None and depth >= self.max_depth:
            return {"leaf": True, "prediction": prediction}

        if len(y) < self.min_samples_split:
            return {"leaf": True, "prediction": prediction}


        best_feature,best_threshold,best_gain = self.find_best_split(X,y)

        if best_feature is None or best_gain <= 0:
            return {"leaf": True, "prediction": prediction}

        left_mask = X[:, best_feature] <= best_threshold
        right_mask = ~left_mask

        left_child = self.build_tree(
            X[left_mask],
            y[left_mask],
            depth+1
        )
        right_child = self.build_tree(
            X[right_mask],
            y[right_mask],
            depth+1
        )

        return {
            "leaf": False,
            "feature":best_feature,
            "threshold":best_threshold,
            "left":left_child,
            "right":right_child
        }


    def predict_one(self, x, node=None):

        if node is None:
            node = self.tree

        if node["leaf"]:
            return node["prediction"]

        feature = node["feature"]
        threshold = node["threshold"]

        if x[feature] <= threshold:
            return self.predict_one(x, node["left"])
        else:
            return self.predict_one(x, node["right"])

    def predict(self, X):
        return np.array([
            self.predict_one(x) for x in X
        ])

        

In [19]:
X = np.array([
    [2], [3], [4],
    [7], [8], [9]
])

y = np.array([0, 0, 0, 1, 1, 1])

tree = Decision_Tree(max_depth=3, min_samples_split=2)

tree.tree = tree.build_tree(X, y)

predictions = tree.predict(X)

print("Predictions:", predictions)
print("Actual:     ", y)
print("Accuracy:   ", np.mean(predictions == y))


X_test = np.array([[1], [5], [6], [10]])
predictions = tree.predict(X_test)

print(predictions)
# Expected: [0 0 1 1]

Predictions: [0 0 0 1 1 1]
Actual:      [0 0 0 1 1 1]
Accuracy:    1.0
[0 0 1 1]
